### Imports

In [1]:
import os
from sklearn.model_selection import train_test_split
import random
import shutil
from pathlib import Path

In [2]:
SOURCE_DIR ="../data/water_dataset"
DEST_DIR = "../data/split_dataset"

CLASSES = ['clean', 'muddy', 'polluted']
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [3]:
# method to create directories 
def create_directories(base_dir, classes):
    for split in ['train', 'test']:
        for class_name in classes:
            os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

In [4]:
random.seed(RANDOM_STATE)

### train-test split

In [5]:
# clean old split
if(os.path.exists(DEST_DIR)):
    shutil.rmtree(DEST_DIR)

# create directories
create_directories(DEST_DIR, CLASSES)

for class_name in CLASSES:
    class_dir = os.path.join(SOURCE_DIR, class_name)

    # final list of image file paths
    images = list(Path(class_dir).glob("*.jpg")) + list(Path(class_dir).glob("*.png"))

    train_imgs, test_imgs = train_test_split(images, test_size= TEST_SIZE, random_state= RANDOM_STATE)

    print("------------Few train images-----------")
    for i in range(5):
        print(train_imgs[i])
    
    print("------------Few test images-----------")
    for i in range(5):
        print(test_imgs[i])

    for img in train_imgs:
        shutil.copy(img, os.path.join(DEST_DIR, 'train', class_name, img.name))

    for img in test_imgs:
        shutil.copy(img, os.path.join(DEST_DIR, 'test', class_name, img.name))

    print(f"{class_name}: {len(train_imgs)} train | {len(test_imgs)} test")

------------Few train images-----------
..\data\water_dataset\clean\177.jpg
..\data\water_dataset\clean\187.jpg
..\data\water_dataset\clean\200.jpg
..\data\water_dataset\clean\213.jpg
..\data\water_dataset\clean\179.jpg
------------Few test images-----------
..\data\water_dataset\clean\191.jpg
..\data\water_dataset\clean\97.jpg
..\data\water_dataset\clean\28.jpg
..\data\water_dataset\clean\63.jpg
..\data\water_dataset\clean\111.jpg
clean: 174 train | 44 test
------------Few train images-----------
..\data\water_dataset\muddy\muddy103.jpg
..\data\water_dataset\muddy\muddy16.jpg
..\data\water_dataset\muddy\muddy45.jpg
..\data\water_dataset\muddy\muddy20.jpg
..\data\water_dataset\muddy\muddy74.jpg
------------Few test images-----------
..\data\water_dataset\muddy\muddy48.jpg
..\data\water_dataset\muddy\muddy24.jpg
..\data\water_dataset\muddy\muddy113.jpg
..\data\water_dataset\muddy\muddy17.jpg
..\data\water_dataset\muddy\muddy61.jpg
muddy: 156 train | 40 test
------------Few train images-

### Preprocessing & DataLoaders

In [6]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

In [7]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

In [8]:
def get_dataloaders(train_dir, test_dir, batch_size):
# preprocessing
    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor()
    ])
    test_transform = transforms.Compose ([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor()
    ])
    # ImageFolder assigns label from folder name
    train_dataset = ImageFolder(train_dir, transform=train_transform) 
    test_dataset = ImageFolder(test_dir, transform=test_transform)


# DataLoader creates batches (batching and loading)
    train_loader = DataLoader( 
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )
    # ========================================add later============================================================
    # num_workers=num_workers,
    # pin_memory=True  # Faster transfer to GPU(batches are stored in page-locked memory & directly transferred to GPU)

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, test_loader

In [9]:
train_loader, test_loader = get_dataloaders('../data/split_dataset/train','../data/split_dataset/test', BATCH_SIZE)